In this notebook, external SMARD data is loaded, cleaned, renamed, and merged with the cleaned day-ahead electricity price data.


In [1]:
import pandas as pd

def convert_smard_numeric_columns(df, timestamp_col="timestamp"):
    df = df.copy()

    for col in df.columns:
        if col != timestamp_col:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(",", "", regex=False)
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

def clean_prepared_dataset(df, timestamp_col="timestamp"):
    df = df.copy()

    df[timestamp_col] = pd.to_datetime(df[timestamp_col])

    df = convert_smard_numeric_columns(df, timestamp_col=timestamp_col)

    df = df.sort_values(timestamp_col)

    df = df.groupby(timestamp_col, as_index=False).mean()

    return df


In [2]:
PRICE_PATH = "../data/processed/clean_price_data.csv"

ACTUAL_LOAD_PATH = "../data/raw/Actual_load_2023_2026.csv"
FORECASTED_LOAD_PATH = "../data/raw/Forecasted_load_2023_2026.csv"

ACTUAL_GENERATION_PATH = "../data/raw/Actual_generation_2023_2026.csv"
FORECASTED_GENERATION_PATH = "../data/raw/Forecasted_generation_2023_2026.csv"

OUTPUT_PATH = "../data/interim/merged_external_dataset.csv"


In [3]:
prices = pd.read_csv(PRICE_PATH)

actual_load = pd.read_csv(ACTUAL_LOAD_PATH, sep=";")
forecasted_load = pd.read_csv(FORECASTED_LOAD_PATH, sep=";")

actual_generation = pd.read_csv(ACTUAL_GENERATION_PATH, sep=";")
forecasted_generation = pd.read_csv(FORECASTED_GENERATION_PATH, sep=";")

In [4]:
print("Prices:", prices.shape)
print("Actual load:", actual_load.shape)
print("Forecasted load:", forecasted_load.shape)
print("Actual generation:", actual_generation.shape)
print("Forecasted generation:", forecasted_generation.shape)

Prices: (27261, 2)
Actual load: (27264, 6)
Forecasted load: (27264, 4)
Actual generation: (27264, 14)
Forecasted generation: (27264, 8)


In [5]:
prices = prices[["timestamp", "price"]].copy()

prices["timestamp"] = pd.to_datetime(prices["timestamp"])

prices = convert_smard_numeric_columns(prices)
prices = clean_prepared_dataset(prices)

prices.head()

,timestamp,price
0,2023-01-01 00:00:00,-5.17
1,2023-01-01 01:00:00,-1.07
2,2023-01-01 02:00:00,-1.47
3,2023-01-01 03:00:00,-5.08
4,2023-01-01 04:00:00,-4.49


In [6]:
actual_load = actual_load[[
    "Start date",
    "grid load [MWh] Calculated resolutions",
    "Residual load [MWh] Calculated resolutions"
]].copy()

actual_load.rename(columns={
    "Start date": "timestamp",
    "grid load [MWh] Calculated resolutions": "actual_load",
    "Residual load [MWh] Calculated resolutions": "actual_residual_load"
}, inplace=True)

actual_load["timestamp"] = pd.to_datetime(actual_load["timestamp"])
actual_load = convert_smard_numeric_columns(actual_load)
actual_load = clean_prepared_dataset(actual_load)

actual_load.head()

C:\Users\ankit\AppData\Local\Temp\ipykernel_15656\811864411.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  actual_load["timestamp"] = pd.to_datetime(actual_load["timestamp"])


,timestamp,actual_load,actual_residual_load
0,2023-01-01 00:00:00,38346.00,6337.75
1,2023-01-01 01:00:00,37777.25,4601.75
2,2023-01-01 02:00:00,36939.75,3581.00
3,2023-01-01 03:00:00,35932.50,4973.75
4,2023-01-01 04:00:00,35486.25,5082.75


In [9]:
actual_load.dtypes

timestamp               datetime64[us]
actual_load                    float64
actual_residual_load           float64
dtype: object

In [10]:
actual_load.shape

(27261, 3)

In [11]:
forecasted_load = forecasted_load[[
    "Start date",
    "grid load [MWh] Calculated resolutions",
    "Residual load [MWh] Calculated resolutions"
]].copy()

forecasted_load.rename(columns={
    "Start date": "timestamp",
    "grid load [MWh] Calculated resolutions": "forecasted_load",
    "Residual load [MWh] Calculated resolutions": "forecasted_residual_load"
}, inplace=True)

forecasted_load["timestamp"] = pd.to_datetime(forecasted_load["timestamp"])
forecasted_load = convert_smard_numeric_columns(forecasted_load)
forecasted_load = clean_prepared_dataset(forecasted_load)

forecasted_load.head()

C:\Users\ankit\AppData\Local\Temp\ipykernel_15656\2051964366.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  forecasted_load["timestamp"] = pd.to_datetime(forecasted_load["timestamp"])


,timestamp,forecasted_load,forecasted_residual_load
0,2023-01-01 00:00:00,41792.50,2798.75
1,2023-01-01 01:00:00,39621.00,886.25
2,2023-01-01 02:00:00,38240.75,-293.50
3,2023-01-01 03:00:00,37205.50,-645.75
4,2023-01-01 04:00:00,37326.75,-2.50


In [12]:
forecasted_load.shape

(27261, 3)

In [13]:
actual_generation = actual_generation[[
    "Start date",
    "Biomass [MWh] Calculated resolutions",
    "Hydropower [MWh] Calculated resolutions",
    "Wind offshore [MWh] Calculated resolutions",
    "Wind onshore [MWh] Calculated resolutions",
    "Photovoltaics [MWh] Calculated resolutions",
    "Nuclear [MWh] Calculated resolutions",
    "Lignite [MWh] Calculated resolutions",
    "Hard coal [MWh] Calculated resolutions",
    "Fossil gas [MWh] Calculated resolutions",
]].copy()

actual_generation.rename(columns={
    "Start date": "timestamp",
    "Biomass [MWh] Calculated resolutions": "actual_biomass",
    "Hydropower [MWh] Calculated resolutions": "actual_hydropower",
    "Wind offshore [MWh] Calculated resolutions": "actual_wind_offshore",
    "Wind onshore [MWh] Calculated resolutions": "actual_wind_onshore",
    "Photovoltaics [MWh] Calculated resolutions": "actual_solar",
    "Nuclear [MWh] Calculated resolutions": "actual_nuclear",
    "Lignite [MWh] Calculated resolutions": "actual_lignite",
    "Hard coal [MWh] Calculated resolutions": "actual_hard_coal",
    "Fossil gas [MWh] Calculated resolutions": "actual_fossil_gas",
}, inplace=True)

actual_generation["timestamp"] = pd.to_datetime(actual_generation["timestamp"])

actual_generation = convert_smard_numeric_columns(actual_generation)
actual_generation = clean_prepared_dataset(actual_generation)

actual_generation["actual_wind_total"] = (
    actual_generation["actual_wind_offshore"] +
    actual_generation["actual_wind_onshore"]
)

actual_generation.head()

C:\Users\ankit\AppData\Local\Temp\ipykernel_15656\185286670.py:27: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  actual_generation["timestamp"] = pd.to_datetime(actual_generation["timestamp"])


,timestamp,actual_biomass,actual_hydropower,actual_wind_offshore,actual_wind_onshore,actual_solar,actual_nuclear,actual_lignite,actual_hard_coal,actual_fossil_gas,actual_wind_total
0,2023-01-01 00:00:00,4014.25,1275.25,3059.25,28947.25,1.75,2459.50,3859.25,2067.50,1593.75,32006.50
1,2023-01-01 01:00:00,3993.25,1226.50,3586.00,29587.50,2.00,2458.75,3866.50,2052.00,1437.00,33173.50
2,2023-01-01 02:00:00,3967.25,1222.50,3842.25,29514.75,1.75,2459.75,3860.25,2034.25,1435.00,33357.00
3,2023-01-01 03:00:00,3973.00,1223.25,3463.25,27493.50,2.00,2460.50,3864.75,2037.00,1432.50,30956.75
4,2023-01-01 04:00:00,3996.50,1244.00,3462.25,26939.00,2.25,2461.00,3841.00,2040.25,1430.75,30401.25


In [14]:
forecasted_generation = forecasted_generation[[
    "Start date",
    "Photovoltaics and wind [MWh] Calculated resolutions",
    "Wind offshore [MWh] Calculated resolutions",
    "Wind onshore [MWh] Calculated resolutions",
    "Photovoltaics [MWh] Calculated resolutions",
]].copy()

forecasted_generation.rename(columns={
    "Start date": "timestamp",
    "Photovoltaics and wind [MWh] Calculated resolutions": "forecasted_solar_wind_total",
    "Wind offshore [MWh] Calculated resolutions": "forecasted_wind_offshore",
    "Wind onshore [MWh] Calculated resolutions": "forecasted_wind_onshore",
    "Photovoltaics [MWh] Calculated resolutions": "forecasted_solar",
}, inplace=True)

forecasted_generation["timestamp"] = pd.to_datetime(forecasted_generation["timestamp"])

forecasted_generation = convert_smard_numeric_columns(forecasted_generation)
forecasted_generation = clean_prepared_dataset(forecasted_generation)

forecasted_generation["forecasted_wind_total"] = (
    forecasted_generation["forecasted_wind_offshore"] +
    forecasted_generation["forecasted_wind_onshore"]
)

forecasted_generation.head()

C:\Users\ankit\AppData\Local\Temp\ipykernel_15656\3203852448.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  forecasted_generation["timestamp"] = pd.to_datetime(forecasted_generation["timestamp"])


,timestamp,forecasted_solar_wind_total,forecasted_wind_offshore,forecasted_wind_onshore,forecasted_solar,forecasted_wind_total
0,2023-01-01 00:00:00,38993.75,3478.25,35515.50,0.0,38993.75
1,2023-01-01 01:00:00,38734.75,3390.25,35344.50,0.0,38734.75
2,2023-01-01 02:00:00,38534.25,3395.50,35138.75,0.0,38534.25
3,2023-01-01 03:00:00,37851.25,3410.25,34441.00,0.0,37851.25
4,2023-01-01 04:00:00,37329.25,3431.25,33898.00,0.0,37329.25


In [15]:
forecasted_generation.shape

(27261, 6)

In [16]:
df = prices.merge(actual_load, on="timestamp", how="left")
df = df.merge(forecasted_load, on="timestamp", how="left")
df = df.merge(actual_generation, on="timestamp", how="left")
df = df.merge(forecasted_generation, on="timestamp", how="left")

df.head()

,timestamp,price,actual_load,actual_residual_load,forecasted_load,forecasted_residual_load,actual_biomass,actual_hydropower,actual_wind_offshore,actual_wind_onshore,...,actual_nuclear,actual_lignite,actual_hard_coal,actual_fossil_gas,actual_wind_total,forecasted_solar_wind_total,forecasted_wind_offshore,forecasted_wind_onshore,forecasted_solar,forecasted_wind_total
0,2023-01-01 00:00:00,-5.17,38346.00,6337.75,41792.50,2798.75,4014.25,1275.25,3059.25,28947.25,...,2459.50,3859.25,2067.50,1593.75,32006.50,38993.75,3478.25,35515.50,0.0,38993.75
1,2023-01-01 01:00:00,-1.07,37777.25,4601.75,39621.00,886.25,3993.25,1226.50,3586.00,29587.50,...,2458.75,3866.50,2052.00,1437.00,33173.50,38734.75,3390.25,35344.50,0.0,38734.75
2,2023-01-01 02:00:00,-1.47,36939.75,3581.00,38240.75,-293.50,3967.25,1222.50,3842.25,29514.75,...,2459.75,3860.25,2034.25,1435.00,33357.00,38534.25,3395.50,35138.75,0.0,38534.25
3,2023-01-01 03:00:00,-5.08,35932.50,4973.75,37205.50,-645.75,3973.00,1223.25,3463.25,27493.50,...,2460.50,3864.75,2037.00,1432.50,30956.75,37851.25,3410.25,34441.00,0.0,37851.25
4,2023-01-01 04:00:00,-4.49,35486.25,5082.75,37326.75,-2.50,3996.50,1244.00,3462.25,26939.00,...,2461.00,3841.00,2040.25,1430.75,30401.25,37329.25,3431.25,33898.00,0.0,37329.25


In [19]:
print("Merged dataset shape:", df.shape)
print()
print(df.info())

Merged dataset shape: (27261, 21)

<class 'pandas.DataFrame'>
RangeIndex: 27261 entries, 0 to 27260
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   timestamp                    27261 non-null  datetime64[us]
 1   price                        27261 non-null  float64       
 2   actual_load                  27261 non-null  float64       
 3   actual_residual_load         27261 non-null  float64       
 4   forecasted_load              27261 non-null  float64       
 5   forecasted_residual_load     27261 non-null  float64       
 6   actual_biomass               27261 non-null  float64       
 7   actual_hydropower            27261 non-null  float64       
 8   actual_wind_offshore         27261 non-null  float64       
 9   actual_wind_onshore          27261 non-null  float64       
 10  actual_solar                 27261 non-null  float64       
 11  actual_nuclear   

In [20]:
missing_values = df.isna().sum().sort_values(ascending=False)

missing_values[missing_values > 0]

actual_nuclear    17794
dtype: int64

In [21]:
duplicate_count = df["timestamp"].duplicated().sum()

print("Number of duplicate timestamps:", duplicate_count)

Number of duplicate timestamps: 0


In [23]:
df = df.drop(columns=["actual_nuclear"])

In [24]:
df.isna().sum().sort_values(ascending=False).head()

timestamp               0
price                   0
actual_load             0
actual_residual_load    0
forecasted_load         0
dtype: int64

In [26]:
df.to_csv("../data/interim/merged_external_dataset.csv", index=False)

print("Final cleaned dataset saved")
print("Final shape:", df.shape)

Final cleaned dataset saved
Final shape: (27261, 20)
